# Flowers-102 — дообучение ResNet-50 и экспорт в ONNX

Ноутбук для **Google Colab с GPU**. На выходе — два файла, которые нужно положить
в папку `model/` проекта:

- `model.onnx` — сеть вместе с нормализацией и softmax внутри графа
- `labels.json` — имена классов, метрики и спецификация препроцессинга

### Перед запуском

`Runtime` → `Change runtime type` → **T4 GPU** (или любой доступный GPU).

Затем `Runtime` → `Run all`. Полный прогон — примерно 15 минут,
из которых большая часть это скачивание датасета (~345 МБ).

### Что делаем

1. Берём `resnet50` с весами ImageNet.
2. **Фаза 1:** backbone заморожен, учим только новый `fc` на 102 класса.
3. **Фаза 2:** размораживаем `layer3` + `layer4`, дообучаем с малым lr.
4. Меряем top-1 на официальном `test` (6149 изображений).
5. Экспортируем в ONNX и сверяем выходы torch и onnxruntime.

Учимся на `train` + `val` (2040 изображений, по 20 на класс) — стандартная
практика для этого датасета, где тестовая часть втрое больше обучающей.

## 0. Окружение

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU НЕ НАЙДЕН: Runtime -> Change runtime type -> T4 GPU"

В Colab нет пакета `onnx`, без которого падает экспортёр, и `onnxruntime`,
нужного для parity-теста. Ставим сразу: если делать это в середине прогона,
pip может попросить перезапуск сессии, а вместе с ним пропадёт обучение.

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.datasets import Flowers102
from torchvision.models import resnet50, ResNet50_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")
if DEVICE == "cpu":
    print("ВНИМАНИЕ: обучение на CPU займёт часы. Включи GPU в настройках runtime.")

## 1. Датасет

`download=True` тянет архив с серверов Оксфорда. В Colab это кэшируется только
до перезапуска сессии — при повторном запуске скачается заново.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224
RESIZE_SIZE = 256

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

ROOT = "./data"
train_part = Flowers102(ROOT, split="train", transform=train_tf, download=True)
val_part = Flowers102(ROOT, split="val", transform=train_tf, download=True)
test_set = Flowers102(ROOT, split="test", transform=eval_tf, download=True)

train_set = ConcatDataset([train_part, val_part])
print(f"train+val: {len(train_set)} | test: {len(test_set)}")

In [ ]:
# Имена классов. В свежем torchvision Flowers102 отдаёт их сам;
# на старых версиях атрибута нет, поэтому берём запасной список.
CLASS_NAMES_FALLBACK = [
    "pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
    "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
    "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
    "yellow iris", "globe-flower", "purple coneflower", "peruvian lily",
    "balloon flower", "giant white arum lily", "fire lily", "pincushion flower",
    "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
    "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox",
    "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya",
    "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy",
    "daffodil", "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold",
    "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula",
    "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium",
    "orange dahlia", "pink-yellow dahlia", "cautleya spicata", "japanese anemone",
    "black-eyed susan", "silverbush", "californian poppy", "osteospermum",
    "spring crocus", "bearded iris", "windflower", "tree poppy", "gazania", "azalea",
    "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
    "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine",
    "desert-rose", "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily",
    "hippeastrum", "bee balm", "ball moss", "foxglove", "bougainvillea", "camellia",
    "mallow", "mexican petunia", "bromelia", "blanket flower", "trumpet creeper",
    "blackberry lily",
]

CLASS_NAMES = list(getattr(test_set, "classes", None) or CLASS_NAMES_FALLBACK)
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 102, f"ожидалось 102 класса, получено {NUM_CLASSES}"
print(f"{NUM_CLASSES} классов, первые пять: {CLASS_NAMES[:5]}")

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

Посмотрим глазами на то, что реально приходит в сеть после аугментаций —
дешёвая страховка от перевёрнутых или обрезанных до неузнаваемости картинок.

In [ ]:
import matplotlib.pyplot as plt

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax in axes.flat:
    img, label = train_set[random.randrange(len(train_set))]
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Модель

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

def set_backbone_trainable(model, names):
    """Размораживает только перечисленные верхнеуровневые блоки (fc всегда обучается)."""
    for param in model.parameters():
        param.requires_grad = False
    for name in list(names) + ["fc"]:
        for param in getattr(model, name).parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"обучаемых параметров: {trainable:,} из {total:,} ({100 * trainable / total:.1f}%)")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

## 3. Цикл обучения и оценки

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return correct / total


def train_phase(model, epochs, lr, trainable_blocks, label):
    set_backbone_trainable(model, trainable_blocks)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))

    for epoch in range(1, epochs + 1):
        model.train()
        started = time.time()
        running_loss = seen = 0
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == "cuda")):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item() * labels.numel()
            seen += labels.numel()
        print(f"[{label}] эпоха {epoch}/{epochs}  loss {running_loss / seen:.4f}"
              f"  {time.time() - started:.0f} c")
    return model

### Фаза 1 — только классификатор

Backbone заморожен, учим новый `fc` с нуля. Это быстро выводит голову из
случайной инициализации, чтобы на следующей фазе большие градиенты не
испортили предобученные признаки.

In [ ]:
train_phase(model, epochs=8, lr=1e-3, trainable_blocks=[], label="фаза 1")
acc_head_only = evaluate(model, test_loader)
print(f"\ntop-1 после фазы 1: {acc_head_only:.4f}")

### Фаза 2 — разморозка верхних блоков

`layer3` и `layer4` отвечают за самые специфичные признаки, их и адаптируем
под цветы. Ранние слои (края, текстуры) оставляем как есть — на 2040
изображениях их дообучение только навредит.

In [ ]:
train_phase(model, epochs=15, lr=1e-4, trainable_blocks=["layer3", "layer4"], label="фаза 2")
acc_final = evaluate(model, test_loader)
print(f"\ntop-1 после фазы 1: {acc_head_only:.4f}")
print(f"top-1 после фазы 2: {acc_final:.4f}")

Сохраняем веса до экспорта: если что-то сломается дальше, обучение
не придётся повторять — состояние можно будет загрузить обратно.

In [ ]:
torch.save(model.state_dict(), "resnet50_flowers102.pt")
print("веса сохранены: resnet50_flowers102.pt")

# Восстановить после перезапуска сессии:
# model.load_state_dict(torch.load("resnet50_flowers102.pt"))

## 4. Экспорт в ONNX

Ключевое решение: **нормализация ImageNet и softmax лежат внутри графа**.
Сайт работает без torch и torchvision, поэтому любое расхождение в
препроцессинге пришлось бы ловить уже в продакшене. Договор с сервером
становится максимально простым: подай RGB-тензор `1x3x224x224` со значениями
в `[0, 1]` — остальное сделает модель.

In [ ]:
class ExportWrapper(nn.Module):
    """Принимает RGB в [0,1], внутри нормализует и отдаёт вероятности."""

    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, pixel_values):
        normalized = (pixel_values - self.mean) / self.std
        return torch.softmax(self.model(normalized), dim=1)


wrapper = ExportWrapper(model, IMAGENET_MEAN, IMAGENET_STD).to("cpu").eval()
dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)

ONNX_PATH = "flowers102.onnx"
export_kwargs = dict(
    input_names=["pixel_values"],
    output_names=["probabilities"],
    opset_version=17,
    dynamic_axes={"pixel_values": {0: "batch"}, "probabilities": {0: "batch"}},
)
try:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, **export_kwargs)

print(f"{ONNX_PATH}: {Path(ONNX_PATH).stat().st_size / 1e6:.1f} МБ")

### Parity-тест

Прогоняем одни и те же изображения через torch и через onnxruntime. Если
вероятности совпадают, экспорт корректен и про torch можно забыть.

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Денормализуем батч из теста обратно в [0,1] — ровно то, что подаст сервер.
images, labels = next(iter(test_loader))
batch = (images[:8] * std + mean).clamp(0, 1)

with torch.no_grad():
    torch_probs = wrapper(batch).numpy()
onnx_probs = session.run(None, {"pixel_values": batch.numpy()})[0]

max_diff = np.abs(torch_probs - onnx_probs).max()
print(f"максимальное расхождение вероятностей: {max_diff:.2e}")
assert max_diff < 1e-4, "torch и onnxruntime расходятся — экспорт сломан"

print("\nпредсказания onnxruntime:")
for i in range(len(batch)):
    top = onnx_probs[i].argmax()
    mark = "+" if top == labels[i].item() else "-"
    print(f"  {mark} {CLASS_NAMES[top]:<28} {onnx_probs[i][top]:.3f}"
          f"   (истина: {CLASS_NAMES[labels[i].item()]})")

## 5. Метаданные и выгрузка

`labels.json` описывает всё, что серверу нужно знать о модели, включая
препроцессинг — чтобы при смене архитектуры не пришлось править код сайта.

In [ ]:
meta = {
    "architecture": "resnet50",
    "dataset": "Flowers102",
    "num_classes": NUM_CLASSES,
    "classes": CLASS_NAMES,
    "accuracy": {
        "test_top1_head_only": round(acc_head_only, 4),
        "test_top1_finetuned": round(acc_final, 4),
        "test_size": len(test_set),
    },
    "preprocessing": {
        "resize_shorter_side": RESIZE_SIZE,
        "center_crop": IMG_SIZE,
        "scale": "divide by 255, range [0,1]",
        "normalization": "inside the ONNX graph",
        "input_name": "pixel_values",
        "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
        "output_name": "probabilities",
        "output": "softmax probabilities",
    },
}

Path("flowers102.labels.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in meta.items() if k != "classes"}, indent=2, ensure_ascii=False))

In [ ]:
from google.colab import files

files.download("flowers102.onnx")
files.download("flowers102.labels.json")

Оба файла нужно положить в папку `model/` проекта:

```
model/
  model.onnx
  labels.json
```

Если Colab блокирует скачивание (бывает в Safari и при строгих настройках
cookies), сохрани на Диск вместо этого:

```python
from google.colab import drive
drive.mount("/content/drive")
!cp model.onnx labels.json /content/drive/MyDrive/
```